# LLM Preference Elicitation: Learning Internal Representations

## 1. Introduction & Methodology

This notebook documents our **LLM Preference Elicitation** experiments using the Generalized Rank-Utility Model (GRUM). 

### Agent Representation: The Embedding Choice
We treat different prompt contexts as individual agents and represent their "state" as a feature vector $x_n$, derived using two strategies:
- **Hidden States (HS)**: PCA-reduced activations from the LLM's last layer. Captures the model's internal latent state.
- **Sentence Transformers (ST)**: PCA-reduced semantic embeddings of the prompt text. Captures external semantic meaning.

### Domain: Colors
We use the **Colors** domain (Blue, Red, Green, Purple, Yellow) to validate the model's ability to learn both intrinsic preferences ($\delta$) and personalized interaction patterns ($B$).

In [1]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import sys
import os
from scipy.stats import pearsonr
from sklearn.metrics.pairwise import cosine_similarity

# Set project root to import grums modules if needed
ROOT = Path.cwd().parent
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

sns.set_theme(style="whitegrid", palette="muted")

# Global Configuration
color_names = ["blue", "red", "green", "purple", "yellow"]
color_palette = {"blue": "#1f77b4", "red": "#d62728", "green": "#2ca02c", "purple": "#9467bd", "yellow": "#d6d627"}

## 2. Load Data & Convergence Analysis
We load results from the latest runs and analyze how the model parameters evolve over the adaptive elicitation steps.

In [2]:
EXP_DIR = ROOT / "results" / "llm" / "llm_colors-20260326-114531"
output_dir = EXP_DIR / "outputs"

results_data = []
raw_results = []
convergence_data = []

if output_dir.exists():
    for f in sorted(output_dir.glob("*.json")):
        with open(f, "r") as j:
            res = json.load(j)
            criterion = res.get("criterion", "social").capitalize()
            embedding = res.get("embedding_method", "")
            if not embedding: embedding = "HS" if "hidden_state" in f.name else "ST"
            else: embedding = "HS" if "hidden_state" in embedding else "ST"
            
            model_full = res.get("model_id", "").split("/")[-1]
            model_type = "Instruct" if "Instruct" in model_full else "Pretrained"
            
            history = res.get("history", {})
            if not history: continue
            
            steps = sorted(map(int, history.keys()))
            last_step = str(steps[-1])
            final_delta = np.array(history[last_step]["grum"]["delta"])
            final_B = np.array(history[last_step]["grum"]["interaction"])
            
            raw_results.append({"Model": model_type, "Embedding": embedding, "Criterion": criterion, "delta": final_delta, "B": final_B})
            for i, color in enumerate(color_names):
                results_data.append({"Model Type": model_type, "Embedding": embedding, "Color": color, "Score (Delta)": final_delta[i]})
            
            # Calculate Convergence & Drift
            prev_delta, prev_B = None, None
            for s in steps:
                curr = history[str(s)]["grum"]
                curr_delta = np.array(curr["delta"])
                curr_B = np.array(curr["interaction"])
                
                mse_final = np.mean((curr_delta - final_delta)**2)
                drift = np.linalg.norm(curr_delta - prev_delta) if prev_delta is not None else 0
                
                convergence_data.append({
                    "Step": s, "Embedding": embedding, "Model": model_type, 
                    "MSE to Final": mse_final, "Parameter Drift": drift
                })
                prev_delta, prev_B = curr_delta, curr_B

df = pd.DataFrame(results_data)
conv_df = pd.DataFrame(convergence_data)
print(f"Loaded {len(raw_results)} experiment runs.")

### 2.1 Convergence and Parameter Drift
We visualize two metrics:
1.  **MSE to Final Estimate**: Shows how close the current parameters are to the end-of-run state. Note that a stochastic estimator (like MC-EM) will typically show a sudden drop at the very end because the "final" state is one realization of the random walk.
2.  **Parameter Drift ($|\theta_t - \theta_{t-1}|$ )**: Shows the step-to-step stability. A decaying drift indicates the model has reached a stationary state.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
sns.lineplot(data=conv_df, x="Step", y="Parameter Drift", hue="Embedding", ax=axes[0])
axes[0].set_title("Step-to-Step Parameter Drift (Stability Test)")
axes[0].set_yscale('log')

sns.lineplot(data=conv_df, x="Step", y="MSE to Final", hue="Embedding", ax=axes[1])
axes[1].set_title("Stochastic Convergence: MSE to Final Estimate")
axes[1].set_yscale('log')
plt.tight_layout()
plt.show()

## 3. Interaction Matrix ($B$) Analysis
We analyze the learned interaction parameters. We focus on **Column Similarity** to distinguish between global agent bias and item-specific personalization.

In [ ]:
sim_results = []
for r in raw_results:
    B = r['B']
    col_sim = cosine_similarity(B.T)
    avg_col_sim = (np.sum(col_sim) - B.shape[1]) / (B.shape[1] * (B.shape[1] - 1))
    sim_results.append({"Model": r['Model'], "Embedding": r['Embedding'], "Avg Col Sim": avg_col_sim})

display(pd.DataFrame(sim_results).groupby(["Model", "Embedding"])[["Avg Col Sim"]].mean().round(4))

### 3.1 Interaction Patterns (Heatmaps)
Visualizing the $B$ matrix for both model types across embeddings.

In [ ]:
def plot_b_matrix(B, title, ax):
    vmax = max(0.5, np.abs(B).max())
    sns.heatmap(B, annot=False, cmap="RdBu_r", center=0, ax=ax, vmax=vmax, vmin=-vmax,
                xticklabels=color_names, yticklabels=[f"F{i}" for i in range(B.shape[0])])
    ax.set_title(title)

for m_type in ["Pretrained", "Instruct"]:
    filtered = [r for r in raw_results if r['Model'] == m_type]
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    for j, emb in enumerate(["HS", "ST"]):
        m = [r for r in filtered if r['Embedding'] == emb and r['Criterion'] == 'Social']
        if m: plot_b_matrix(m[0]['B'], f"{m_type} | {emb}", axes[j])
        else: axes[j].axis('off')
    plt.suptitle(f"Interaction Matrix Overview: {m_type} Model", fontsize=14)
    plt.tight_layout(rect=[0, 0, 1, 0.95])
    plt.show()

## 4. Preference Archetypes (Deltas)
Comparing the underlying rank-order of colors across model types and embedding methods.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
sns.barplot(data=df, x="Color", y="Score (Delta)", hue="Model Type", ax=axes[0], errorbar=None, palette="Set1")
axes[0].set_title("Model Strategy: Pretrained vs Instruct")
sns.barplot(data=df, x="Color", y="Score (Delta)", hue="Embedding", ax=axes[1], errorbar=None, palette="Set2")
axes[1].set_title("Embedding Consistency")
for ax in axes: ax.axhline(0, color='black', alpha=0.3)
plt.tight_layout()
plt.show()

**Final Insights**
- **Convergence**: The high initial Parameter Drift decays rapidly, settling into a stochastic steady state by round 200. The "jump" at the very end of the MSE plot is a numerical byproduct of the MC-EM stochasticity being compared to its own final sample.
- **Personalization**: HS embeddings continue to show high column similarity, indicating they capture global response characteristics. ST embeddings are more discriminative of item-specific persona preferences.